# Practice 1 - CNNs: Part 2 - Pretrained CNNs

**Authors:** [Name 1], [Name 2]

**Date:** March 2026

**Description:** In this notebook we use several pretrained CNN models (transfer learning) on the STL-10 dataset, applying both feature extraction and fine-tuning strategies.

---
## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Dataset Loading & Preprocessing](#2-dataset-loading--preprocessing)
3. [Strategy 1: Feature Extraction](#3-strategy-1-feature-extraction)
    - 3.1 VGG16
    - 3.2 ResNet50
    - 3.3 MobileNetV2
4. [Strategy 2: Fine-Tuning](#4-strategy-2-fine-tuning)
    - 4.1 VGG16 Fine-Tuned
    - 4.2 ResNet50 Fine-Tuned
    - 4.3 MobileNetV2 Fine-Tuned
5. [Results Comparison](#5-results-comparison)
6. [Comparison with Custom CNNs](#6-comparison-with-custom-cnns)
7. [Conclusions](#7-conclusions)

---
## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from keras import layers, Model
from keras import applications
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Constants
IMG_SIZE = 96        # STL-10 native resolution
NUM_CLASSES = 10
BATCH_SIZE = 32
EPOCHS_FE = 30       # Feature extraction epochs
EPOCHS_FT = 30       # Fine-tuning epochs
VALIDATION_SPLIT = 0.2

# Directory to save models
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

---
## 2. Dataset Loading & Preprocessing

We load STL-10 and prepare the data pipelines. For pretrained models, we use the specific preprocessing function provided by each model (e.g., `preprocess_input`).

In [ ]:
# Load STL-10
(ds_train, ds_test), ds_info = tfds.load(
    'stl10',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

class_names = ds_info.features['label'].names
print(f"Classes: {class_names}")
print(f"Training samples: {ds_info.splits['train'].num_examples}")
print(f"Test samples: {ds_info.splits['test'].num_examples}")

In [ ]:
def preprocess(image, label):
    """Normalize images to [0, 1] and resize."""
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    return image, label

# Train/validation split
num_train = ds_info.splits['train'].num_examples
num_val = int(num_train * VALIDATION_SPLIT)

ds_train_shuffled = ds_train.shuffle(num_train, seed=SEED)
ds_val = ds_train_shuffled.take(num_val)
ds_train_split = ds_train_shuffled.skip(num_val)

train_dataset = (
    ds_train_split
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(num_train)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    ds_val
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    ds_test
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

---
## Helper Functions

In [ ]:
def plot_history(history, title=""):
    """Plot training and validation accuracy and loss curves."""
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs_range, acc, 'b-o', label='Training Accuracy')
    ax1.plot(epochs_range, val_acc, 'r--o', label='Validation Accuracy')
    ax1.set_title(f'{title} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, loss, 'b-o', label='Training Loss')
    ax2.plot(epochs_range, val_loss, 'r--o', label='Validation Loss')
    ax2.set_title(f'{title} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def train_and_evaluate(model, model_name, train_ds, val_ds, test_ds, epochs):
    """Train a model, plot history, evaluate on test set, and return results."""
    
    filepath = os.path.join(MODEL_DIR, f"{model_name}.keras")
    
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=filepath,
            save_best_only=True,
            monitor='val_loss',
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]

    history = model.fit(
        train_ds,
        epochs=epochs,
        validation_data=val_ds,
        callbacks=callbacks
    )

    plot_history(history, title=model_name)

    best_model = keras.models.load_model(filepath)
    test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
    print(f"\n{model_name} - Test Accuracy: {test_acc:.4f}, Test Loss: {test_loss:.4f}")

    return {
        'model_name': model_name,
        'history': history,
        'test_accuracy': test_acc,
        'test_loss': test_loss
    }

In [ ]:
# Store results for comparison
results = []

---
## 3. Strategy 1: Feature Extraction

In **feature extraction**, we use the convolutional base of a pretrained model as a fixed feature extractor. The pretrained layers are **frozen** (not trainable) and we only train a new classification head on top.

**Advantages:**
- Very fast training (only the classifier head is trained)
- Works well even with small datasets
- Leverages features learned from ImageNet (millions of images)

### 3.1 VGG16 - Feature Extraction

In [ ]:
def build_vgg16_feature_extraction(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """VGG16 with frozen base for feature extraction."""
    
    # Load pretrained VGG16 without the classification head
    base_model = applications.VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all layers in the base model
    base_model.trainable = False
    
    # Build the full model
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.vgg16.preprocess_input(x)  # VGG16-specific preprocessing
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='vgg16_feature_extraction')
    return model

vgg16_fe_model = build_vgg16_feature_extraction()
vgg16_fe_model.summary()

In [ ]:
# Print trainable vs non-trainable parameters
print(f"Trainable parameters: {sum(p.size for p in vgg16_fe_model.trainable_weights):,}")
print(f"Non-trainable parameters: {sum(p.size for p in vgg16_fe_model.non_trainable_weights):,}")

In [ ]:
vgg16_fe_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_vgg16_fe = train_and_evaluate(
    vgg16_fe_model, 'vgg16_feature_extraction',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FE
)
results.append(result_vgg16_fe)

**Observations (VGG16 Feature Extraction):**

_TODO: Comment on training speed and accuracy. How does this compare to training from scratch?_

### 3.2 ResNet50 - Feature Extraction

In [ ]:
def build_resnet50_feature_extraction(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """ResNet50 with frozen base for feature extraction."""
    
    base_model = applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.resnet50.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='resnet50_feature_extraction')
    return model

resnet50_fe_model = build_resnet50_feature_extraction()
resnet50_fe_model.summary()

In [ ]:
resnet50_fe_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_resnet50_fe = train_and_evaluate(
    resnet50_fe_model, 'resnet50_feature_extraction',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FE
)
results.append(result_resnet50_fe)

**Observations (ResNet50 Feature Extraction):**

_TODO: Comment on results and compare with VGG16 feature extraction._

### 3.3 MobileNetV2 - Feature Extraction

In [ ]:
def build_mobilenetv2_feature_extraction(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """MobileNetV2 with frozen base for feature extraction."""
    
    base_model = applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False
    
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='mobilenetv2_feature_extraction')
    return model

mobilenet_fe_model = build_mobilenetv2_feature_extraction()
mobilenet_fe_model.summary()

In [ ]:
mobilenet_fe_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_mobilenet_fe = train_and_evaluate(
    mobilenet_fe_model, 'mobilenetv2_feature_extraction',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FE
)
results.append(result_mobilenet_fe)

**Observations (MobileNetV2 Feature Extraction):**

_TODO: Comment on results. Discuss parameter efficiency of MobileNetV2 compared to VGG16 and ResNet50._

---
## 4. Strategy 2: Fine-Tuning

In **fine-tuning**, we unfreeze a few of the top layers of the pretrained base and jointly train both the newly added classifier layers and the unfrozen top layers. This allows us to slightly adjust the high-level representations to make them more relevant for our specific task.

**Important:** Fine-tuning should be done with a very low learning rate to avoid destroying the pretrained features.

**Procedure:**
1. First train the model with the base frozen (feature extraction phase) — already done above.
2. Then unfreeze the top layers of the base and continue training with a lower learning rate.

### 4.1 VGG16 - Fine-Tuning

In [ ]:
def build_vgg16_fine_tuning(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES, 
                            unfreeze_from='block5_conv1'):
    """VGG16 with partial unfreezing for fine-tuning."""
    
    base_model = applications.VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all layers first
    base_model.trainable = True
    
    # Freeze everything before the specified layer
    set_trainable = False
    for layer in base_model.layers:
        if layer.name == unfreeze_from:
            set_trainable = True
        layer.trainable = set_trainable
    
    # Print which layers are trainable
    for layer in base_model.layers:
        print(f"{layer.name}: trainable={layer.trainable}")
    
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.vgg16.preprocess_input(x)
    x = base_model(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='vgg16_fine_tuning')
    return model

vgg16_ft_model = build_vgg16_fine_tuning()
vgg16_ft_model.summary()

In [ ]:
# Use a lower learning rate for fine-tuning
vgg16_ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_vgg16_ft = train_and_evaluate(
    vgg16_ft_model, 'vgg16_fine_tuning',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FT
)
results.append(result_vgg16_ft)

**Observations (VGG16 Fine-Tuning):**

_TODO: Compare with VGG16 feature extraction. Did fine-tuning improve accuracy? Discuss the importance of using a low learning rate._

### 4.2 ResNet50 - Fine-Tuning

In [ ]:
def build_resnet50_fine_tuning(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES,
                               num_layers_to_unfreeze=20):
    """ResNet50 with the last N layers unfrozen for fine-tuning."""
    
    base_model = applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all layers first, then unfreeze the last N
    base_model.trainable = True
    for layer in base_model.layers[:-num_layers_to_unfreeze]:
        layer.trainable = False
    
    # Count trainable layers
    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    print(f"Trainable layers in base: {trainable_count}/{len(base_model.layers)}")
    
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.resnet50.preprocess_input(x)
    x = base_model(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='resnet50_fine_tuning')
    return model

resnet50_ft_model = build_resnet50_fine_tuning()
resnet50_ft_model.summary()

In [ ]:
resnet50_ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_resnet50_ft = train_and_evaluate(
    resnet50_ft_model, 'resnet50_fine_tuning',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FT
)
results.append(result_resnet50_ft)

**Observations (ResNet50 Fine-Tuning):**

_TODO: Compare with ResNet50 feature extraction. Comment on the improvement._

### 4.3 MobileNetV2 - Fine-Tuning

In [ ]:
def build_mobilenetv2_fine_tuning(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES,
                                  num_layers_to_unfreeze=30):
    """MobileNetV2 with the last N layers unfrozen for fine-tuning."""
    
    base_model = applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    base_model.trainable = True
    for layer in base_model.layers[:-num_layers_to_unfreeze]:
        layer.trainable = False
    
    trainable_count = sum(1 for l in base_model.layers if l.trainable)
    print(f"Trainable layers in base: {trainable_count}/{len(base_model.layers)}")
    
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='mobilenetv2_fine_tuning')
    return model

mobilenet_ft_model = build_mobilenetv2_fine_tuning()
mobilenet_ft_model.summary()

In [ ]:
mobilenet_ft_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

result_mobilenet_ft = train_and_evaluate(
    mobilenet_ft_model, 'mobilenetv2_fine_tuning',
    train_dataset, val_dataset, test_dataset,
    epochs=EPOCHS_FT
)
results.append(result_mobilenet_ft)

**Observations (MobileNetV2 Fine-Tuning):**

_TODO: Compare with MobileNetV2 feature extraction. Discuss improvement and efficiency._

---
## 5. Results Comparison

Summary of all pretrained models and strategies.

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame([
    {
        'Model': r['model_name'],
        'Test Accuracy': f"{r['test_accuracy']:.4f}",
        'Test Loss': f"{r['test_loss']:.4f}"
    }
    for r in results
])

print("\n" + "="*60)
print("RESULTS SUMMARY - Pretrained CNNs")
print("="*60)
display(comparison_df)

In [ ]:
# Bar chart comparison
model_names = [r['model_name'] for r in results]
test_accs = [r['test_accuracy'] for r in results]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

plt.figure(figsize=(14, 5))
bars = plt.bar(range(len(model_names)), test_accs, color=colors[:len(model_names)])
plt.xticks(range(len(model_names)), model_names, rotation=30, ha='right')
plt.ylabel('Test Accuracy')
plt.title('Pretrained CNN Models - Test Accuracy Comparison')
plt.ylim(0, 1)
for bar, acc in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Comparison with Custom CNNs

Load the results from Part 1 (custom CNNs) and compare with the pretrained models.

_TODO: Manually add the test accuracies from Part 1 here, or load them programmatically._

In [ ]:
# TODO: Fill in the results from Part 1 (Custom CNNs notebook)
custom_results = [
    {'model_name': 'baseline_cnn', 'test_accuracy': 0.0, 'test_loss': 0.0},
    {'model_name': 'improved_cnn', 'test_accuracy': 0.0, 'test_loss': 0.0},
    {'model_name': 'custom_resnet', 'test_accuracy': 0.0, 'test_loss': 0.0},
    {'model_name': 'custom_inception', 'test_accuracy': 0.0, 'test_loss': 0.0},
]

all_results = custom_results + results

all_df = pd.DataFrame([
    {
        'Model': r['model_name'],
        'Type': 'Custom' if r in custom_results else 'Pretrained',
        'Test Accuracy': f"{r['test_accuracy']:.4f}",
        'Test Loss': f"{r['test_loss']:.4f}"
    }
    for r in all_results
])

print("\n" + "="*70)
print("FULL RESULTS SUMMARY - Custom vs Pretrained CNNs")
print("="*70)
display(all_df)

In [ ]:
# Combined bar chart
all_names = [r['model_name'] for r in all_results]
all_accs = [r['test_accuracy'] for r in all_results]
all_colors = ['steelblue' if r in custom_results else 'coral' for r in all_results]

plt.figure(figsize=(16, 6))
bars = plt.bar(range(len(all_names)), all_accs, color=all_colors)
plt.xticks(range(len(all_names)), all_names, rotation=40, ha='right')
plt.ylabel('Test Accuracy')
plt.title('All Models - Test Accuracy Comparison (Blue=Custom, Red=Pretrained)')
plt.ylim(0, 1)
for bar, acc in zip(bars, all_accs):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
plt.tight_layout()
plt.show()

---
## 7. Conclusions

_TODO: Write a comprehensive analysis covering:_

- _Which pretrained model and strategy achieved the best results?_
- _Feature extraction vs fine-tuning: when does each strategy work best?_
- _How do pretrained models compare to custom CNNs from Part 1?_
- _What is the benefit of transfer learning, especially with small datasets?_
- _Advantages and disadvantages of each pretrained architecture (VGG16, ResNet50, MobileNetV2)._
- _Practical considerations: training time, model size, ease of use._
- _What further improvements could be explored?_